In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BASE_DIR = Path('/cta/users/guneyn23')
TIMEPOINT_DIR = BASE_DIR / 'relative_repair/rpkm/noUV_atac_damage_all'
OUTPUT_DIR = BASE_DIR / 'relative_repair/window_mean_first_relative'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUMBER_OF_WINDOWS = 400
TIMEPOINTS = ['15m', '30m', '1h', '4h', '8h']

REFERENCE_FILES = {
    'CPD': {
        'real': BASE_DIR / 'rpkm/CPD_rpkm/atac_rpkm/real_damage_rpkm.bed',
        'sim': BASE_DIR / 'rpkm/CPD_rpkm/atac_rpkm/sim_damage_rpkm.bed',
    },
    '64': {
        'real': BASE_DIR / 'rpkm/64_rpkm/ATAC_real_64_rpkm.bed',
        'sim': BASE_DIR / 'rpkm/64_rpkm/ATAC_simulated_64_rpkm.bed',
    },
}

TIMEPOINT_FILES = {
    ('CPD', '15m'): 'R3Hela_15mCPD_TAGCTT_S2_hg38_primary_assembly_DS',
    ('CPD', '30m'): 'R3Hela_30mCPD_GGCTAC_S8_hg38_primary_assembly_DS',
    ('CPD', '1h'): 'R3Hela_1hCPD_CTTGTA_S4_hg38_primary_assembly_DS',
    ('CPD', '4h'): 'R3Hela_4hCPD_AGTCAA_S10_hg38_primary_assembly_DS',
    ('CPD', '8h'): 'R3Hela_8hCPD_AGTTCC_S12_hg38_primary_assembly_DS',
    ('64', '15m'): 'R3Hela_15m64_TTAGGC_S1_hg38_primary_assembly_DS',
    ('64', '30m'): 'R3Hela_30m64_TGACCA_S7_hg38_primary_assembly_DS',
    ('64', '1h'): 'R3Hela_1h64_ACAGTG_S3_hg38_primary_assembly_DS',
    ('64', '4h'): 'R3Hela_4h64_GCCAAT_S9_hg38_primary_assembly_DS',
    ('64', '8h'): 'R3Hela_8h64_CAGATC_S11_hg38_primary_assembly_DS',
}

print(f'Output: {OUTPUT_DIR}')

Output: /cta/users/guneyn23/relative_repair/window_mean_first_relative


In [2]:
def timepoint_path(damage_type, timepoint, kind):
    """Return the existing real/sim timepoint RPKM file."""
    stem = TIMEPOINT_FILES[(damage_type, timepoint)]
    sim_part = '_sim' if kind == 'sim' else ''
    return TIMEPOINT_DIR / f'{stem}{sim_part}_noUV_ATAC_400windows_rpkm.bed'



In [ ]:
def window_mean_rpkm(path, number_of_windows=NUMBER_OF_WINDOWS):
    data = pd.read_csv(path, sep='\t', header=None)

    data['rpkm'] = data.iloc[:, -1]
    data['window'] = (data.index % number_of_windows) + 1
    data['window_size_bp'] = data.iloc[:, 2] - data.iloc[:, 1]

  

    mean_rpkm = data.groupby('window')['rpkm'].mean()
    counts = data.groupby('window')['rpkm'].count()

    return pd.DataFrame({
        'window': mean_rpkm.index,
        'window_size_bp': int(window_sizes[0]),
        'mean_rpkm': mean_rpkm.values,
        'n': counts.values,
    })


window_mean_rpkm(REFERENCE_FILES['CPD']['real']).head()

In [ ]:
def calculate_one(damage_type, timepoint):
    ref_real = window_mean_rpkm(REFERENCE_FILES[damage_type]['real'])
    ref_sim = window_mean_rpkm(REFERENCE_FILES[damage_type]['sim'])
    later_real = window_mean_rpkm(timepoint_path(damage_type, timepoint, 'real'))
    later_sim = window_mean_rpkm(timepoint_path(damage_type, timepoint, 'sim'))

    window_sizes = {
        int(ref_real['window_size_bp'].iloc[0]), int(ref_sim['window_size_bp'].iloc[0]),
        int(later_real['window_size_bp'].iloc[0]), int(later_sim['window_size_bp'].iloc[0]),
    }
    if len(window_sizes) != 1:
        raise ValueError(f'{damage_type} {timepoint}: input window sizes differ: {sorted(window_sizes)}')
    window_size_bp = window_sizes.pop()

    result = pd.DataFrame({
        'damage_type': damage_type,
        'timepoint': timepoint,
        'window': ref_real['window'],
        'window_size_bp': window_size_bp,
        'distance_kb': ((ref_real['window'] - 0.5) * window_size_bp - NUMBER_OF_WINDOWS * window_size_bp / 2) / 1000,
        'reference_real_mean_rpkm': ref_real['mean_rpkm'],
        'timepoint_real_mean_rpkm': later_real['mean_rpkm'],
        'reference_sim_mean_rpkm': ref_sim['mean_rpkm'],
        'timepoint_sim_mean_rpkm': later_sim['mean_rpkm'],
        'n_reference_real': ref_real['n'],
        'n_timepoint_real': later_real['n'],
        'n_reference_sim': ref_sim['n'],
        'n_timepoint_sim': later_sim['n'],
    })

    result['relative_real'] = (
        result['reference_real_mean_rpkm'] - result['timepoint_real_mean_rpkm']
    ) / result['reference_real_mean_rpkm']

    result['relative_sim'] = (
        result['reference_sim_mean_rpkm'] - result['timepoint_sim_mean_rpkm']
    ) / result['reference_sim_mean_rpkm']

    result['relative_real_div_relative_sim'] = (
        result['relative_real'] / result['relative_sim']
    )
    return result


all_results = []
for damage_type in REFERENCE_FILES:
    for timepoint in TIMEPOINTS:
        one_result = calculate_one(damage_type, timepoint)
        one_result.to_csv(
            OUTPUT_DIR / f'{damage_type}_{timepoint}_window_mean_relative.tsv',
            sep='\t', index=False, float_format='%.8g'
        )
        all_results.append(one_result)

summary = pd.concat(all_results, ignore_index=True)
summary_file = OUTPUT_DIR / 'all_window_mean_relative_real_div_sim.tsv'
summary.to_csv(summary_file, sep='\t', index=False, float_format='%.8g')
print(f'Written: {summary_file}')
summary.head()

In [ ]:
colors = {'15m': '#ff595e', '30m': '#ffca3a', '1h': '#8ac926', '4h': '#1982c4', '8h': '#6a4c93'}

for damage_type in REFERENCE_FILES:
    fig, ax = plt.subplots(figsize=(11, 5))
    subset = summary[summary['damage_type'] == damage_type]

    for timepoint in TIMEPOINTS:
        data = subset[subset['timepoint'] == timepoint]
        ax.plot(data['distance_kb'], data['relative_real_div_relative_sim'],
                label=timepoint, color=colors[timepoint], linewidth=1.5)

    ax.axvline(0, color='gray', linestyle='--', linewidth=1)
    ax.set(xlabel='Distance from ATAC peak center (kb)',
           ylabel='Relative real / relative sim',
           title=f'{damage_type}: window-mean relative real / relative sim',
           xlim=(-10, 10))
    ax.grid(True, linestyle='--', alpha=0.35)
    ax.legend(title='Time')
    fig.tight_layout()
    plot_file = OUTPUT_DIR / f'{damage_type}_window_mean_relative_real_div_sim.png'
    fig.savefig(plot_file, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Written: {plot_file}')